# v3 — Memetic Algorithm: Multiple CDPs

The Memetic Algorithm on **several CDPs along the line** of the synthetic FD model,
with and without free-surface multiples: one run per case, same seed and
budget everywhere. This is a **visual check** — where the picks land on
each panel and how they line up from CDP to CDP. The statistical
comparison of the algorithms over these cases is done in the terminal:

    python scripts/run_batch.py configs/batch_fd_multicdp_v3_sep200.json --workers 4
    python scripts/run_batch.py configs/batch_fd_multicdp_v3_sep200_grad200.json --workers 4

The model has gently dipping interfaces and a normal fault at x = 3300 m,
so every CDP sees the events at slightly different times and velocities;
the CDPs at 3200 and 3400 m sit on the two sides of the fault. The CDP at
2500 m is the one used to set up the formulation (semblance options,
separation, tau, minimum gradient); **the others were never used to tune
anything**, so they are the held-out test.

Each CDP is picked independently (no information from the neighbors).

**Execution order:**
1. Setup
2. Parameters
3. The model and the CDP positions
4. Load the cases
5. Run on every case
6. Picks over the semblance panels
7. Results per CDP
8. NMO check: gathers before and after correction
9. Picks along the line
10. Summary

The cases come from `python scripts/model_fd.py` (CDP at 2500 m) and
`python scripts/model_fd.py --config configs/synthetic_fd_model_multicdp.json`
(the other five).

## 1. Setup

In [ ]:
import sys
import os
import glob
import json
import re
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import matplotlib.pyplot as plt

# Project root: walk up to the folder that contains src/
from pathlib import Path
project_root = Path.cwd().resolve()
while not (project_root / 'src').is_dir():
    if project_root.parent == project_root:
        raise RuntimeError('Project root (folder containing src/) not found.')
    project_root = project_root.parent
project_root = str(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

FIG_DIR = os.path.join(project_root, 'results', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project root: {project_root}')

In [ ]:
from src.seismic.cases      import load_case
from src.seismic.semblance  import compute_semblance, robust_threshold, ROBUST_SEMBLANCE
from src.seismic.model2d    import build_layered_model
from src.analysis.reference import get_reference, match_reference
from src.analysis.nmo_viz   import plot_nmo_check
from src.optimizers.memetic import MemeticAlgorithm

print('All modules imported successfully.')

## 2. Parameters

In [ ]:
ALG = 'Memetic Algorithm'
TAG = 'ma'

# --- Cases: every CDP of the FD model found in cases/ (fs and nofs) ---
# Set CDPS to a list (e.g. [2500, 3200]) to run only some of them.
CDPS  = None
MODES = {'fs': 'with free surface (multiples)', 'nofs': 'without free surface'}
DEV_CDP = 2500               # CDP used to set up the formulation

# --- Formulation: same as configs/batch_fd_multicdp_v3_ma.json ---
VEL_MIN      = 1400.0
VEL_MAX      = 5000.0
VEL_STEP     = 20.0          # panel used for tau and for the figures
MIN_SEP_MS   = 200.0
T_MAX_MS     = 2150.0        # latest pick: the side-boundary reflection arrives after it
TAU_FACTOR   = 10.0
P_INSERT     = 0.1
P_REMOVE     = 0.1
SEMBLANCE    = ROBUST_SEMBLANCE   # stretch mute 0.5, energy floor 1%, min fold 10
MIN_VEL_GRADIENT = 200.0     # m/s per second between consecutive picks; 0 = monotonic
REFERENCE_ALT_RATIO = 0.9    # a pick on an equivalent maximum of an event counts (None: peak only)
CACHE_SEMBLANCE = True       # same results, faster

# --- Budget and seed (the same for every case) ---
MAX_EVALS    = 100000
SEED         = 42
N_WORKERS    = 4             # cases run in parallel (1 = one after the other)

# --- Local moves ---
STEP_TIME    = 5
STEP_VEL     = 50.0

# --- Figures ---
V_SHOW       = 3000.0        # velocity axis of the panels (m/s)
MODEL_CONFIG = os.path.join(project_root, 'configs', 'synthetic_fd_model.json')
NMO_CDPS     = None          # CDPs shown in the NMO check (None = all; e.g. [2500, 3200])
NMO_STRETCH_MUTE = SEMBLANCE['stretch_mute']   # same mute as the objective

# --- Memetic Algorithm ---
MA_POP_SIZE    = 20          # solutions in the population
MA_TOURNAMENT  = 2           # tournament size of the parent selection
MA_N_CUTS      = 1           # time cuts of the crossover (1: A|B, 2: A|B|A)
MA_P_MUTATION  = 0.5         # probability of one mutation move per child
MA_N_NEIGHBORS = 10          # neighbors per local-search iteration
MA_LS_PATIENCE = 5           # local-search iterations without improvement before it stops
MA_RESTART     = 100         # children in a row with no new solution -> partial restart

# Output names: runs with a minimum gradient are saved apart
if MIN_VEL_GRADIENT > 0:
    TAG = f'{TAG}_g{MIN_VEL_GRADIENT:g}'

## 3. The model and the CDP positions

P-wave velocity of the model with the CDPs found in `cases/` (dashed
lines: the CDP at 2500 m, used for development, in white; the held-out
CDPs in yellow) and the fault (red). Each CDP gather uses sources and
receivers up to 1500 m on each side of its CDP, so the gathers at 3200
and 3400 m have rays crossing the fault.

In [ ]:
def cdp_of(path):
    return int(re.search(r'cdp(\d+)_', os.path.basename(path)).group(1))

found = sorted({cdp_of(p) for p in glob.glob(os.path.join(project_root, 'cases', 'fd_layered_cdp*_fs.npz'))})
CDP_LIST = found if CDPS is None else [c for c in CDPS if c in found]
missing = [] if CDPS is None else [c for c in CDPS if c not in found]
if missing:
    print(f'Not found in cases/: {missing} -> run scripts/model_fd.py with the multi-CDP config')
if len(CDP_LIST) <= 1:
    print('Only', CDP_LIST, 'available. The other CDPs come from:\n'
          '  python scripts/model_fd.py --config configs/synthetic_fd_model_multicdp.json')
CASES = [f'fd_layered_cdp{x:04d}_{m}' for x in CDP_LIST for m in MODES]
print('CDPs:', CDP_LIST)

with open(MODEL_CONFIG) as f:
    model_cfg = json.load(f)['model']
model = build_layered_model(model_cfg)
faults = sorted({itf['fault']['x_m'] for itf in model_cfg['interfaces'] if 'fault' in itf})

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(model['vp'].T, aspect='auto', cmap='viridis',
               extent=[model['x'][0], model['x'][-1], model['z'][-1], model['z'][0]])
plt.colorbar(im, ax=ax, label='Vp (m/s)')
for x in CDP_LIST:
    ax.axvline(x, color='white' if x == DEV_CDP else 'yellow', ls='--', lw=1.2)
    ax.text(x, -40, f'{x}', ha='center', va='bottom', fontsize=9)
for xf in faults:
    ax.axvline(xf, color='red', lw=1.0, alpha=0.7)
ax.set_ylim(model_cfg['depth_m'], 0)
ax.set_xlabel('x (m)'); ax.set_ylabel('Depth (m)')
ax.set_title('Velocity model, CDP positions (white: development, yellow: held-out) and fault (red)',
             pad=18)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'multicdp_model.png'), dpi=130)
plt.show()

## 4. Load the cases

For each case: semblance panel and tau with the same options as the
objective, and the reference picks (semblance peak of each expected
event) for the figures and the scores — `+` white = primary, `+` magenta =
surface multiple. The algorithm never sees the reference.

In [ ]:
velocities = np.arange(VEL_MIN, VEL_MAX + VEL_STEP, VEL_STEP, dtype=np.float32)
data = {}
for name in CASES:
    case  = load_case(os.path.join(project_root, 'cases', f'{name}.npz'))
    panel = compute_semblance(case['gather'], case['offsets'], velocities, case['dt_ms'], **SEMBLANCE)
    data[name] = {
        'gather': case['gather'], 'offsets': case['offsets'], 'dt': case['dt_ms'],
        't_grid': case['t_grid'], 'panel': panel,
        'tau': robust_threshold(panel, factor=TAU_FACTOR),
        'extent': [VEL_MIN, VEL_MAX, case['t_grid'][-1], case['t_grid'][0]],
        'reference': get_reference(case, alt_ratio=REFERENCE_ALT_RATIO, **SEMBLANCE),
    }
    prim = [r for r in data[name]['reference'] if r['kind'] == 'primary']
    print(f"{name:26s} tau = {data[name]['tau']:.3f}   primaries (ms, m/s): "
          + ', '.join(f"({r['t_ms']:.0f}, {r['v']:.0f})" for r in prim))

## 5. Run on every case

One run per case, same seed. With `N_WORKERS > 1` the cases run in
parallel (each run depends only on its seed, so the result is the same as
one after the other; only the times change). Parallel runs need the
`fork` start method (Linux / WSL); elsewhere they run one after the other.

In [ ]:
def make_optimizer(d):
    return MemeticAlgorithm(
        d['gather'], d['offsets'], d['dt'], vel_min=VEL_MIN, vel_max=VEL_MAX,
        min_sep_ms=MIN_SEP_MS, t_max_ms=T_MAX_MS, objective='threshold', tau=d['tau'],
        p_insert=P_INSERT, p_remove=P_REMOVE, seed=SEED, max_evals=MAX_EVALS,
        step_time=STEP_TIME, step_vel=STEP_VEL,
        min_vel_gradient=MIN_VEL_GRADIENT, cache_semblance=CACHE_SEMBLANCE, **SEMBLANCE,
        pop_size=MA_POP_SIZE, tournament_size=MA_TOURNAMENT, n_cuts=MA_N_CUTS,
        p_mutation=MA_P_MUTATION, n_neighbors=MA_N_NEIGHBORS, ls_patience=MA_LS_PATIENCE,
        restart_after=MA_RESTART)


def run_case(name):
    d = data[name]
    opt = make_optimizer(d).run()
    t_s, v_s = opt.get_result()
    t_ms = np.asarray(t_s) * d['dt']
    return name, {'t_ms': t_ms, 'v': np.asarray(v_s, dtype=float), 'score': opt.best_score,
                  'time_s': opt.exec_time_s,
                  'match': match_reference(t_ms, v_s, d['reference'])}


if N_WORKERS > 1 and 'fork' in mp.get_all_start_methods():
    with ProcessPoolExecutor(N_WORKERS, mp_context=mp.get_context('fork')) as pool:
        results = dict(pool.map(run_case, CASES))
else:
    results = dict(run_case(name) for name in CASES)

for name in CASES:
    r, m = results[name], results[name]['match']
    print(f"{name:26s} f = {r['score']:6.3f}   K = {m['n_picks']}   "
          f"on primary {m['n_on_primary']}/{m['n_primaries']}, on multiple {m['n_on_multiple']}, "
          f"F1 {m['f1']:.2f}   ({r['time_s']:.0f} s)")

## 6. Picks over the semblance panels

One column per CDP; top row with free surface, bottom row without.
Picks: white circle = on a primary, magenta triangle = on a surface
multiple, red X = on neither (tolerance 40 ms and 2%, as in the scores).

In [ ]:
STYLE = {'primary': dict(marker='o', color='white', ms=7),
         'multiple': dict(marker='^', color='magenta', ms=8),
         None: dict(marker='X', color='red', ms=8)}


def draw_case(ax, name):
    d, r = data[name], results[name]
    ax.imshow(d['panel'].T, aspect='auto', cmap='jet', vmin=0, vmax=1, extent=d['extent'])
    for ref in d['reference']:
        ax.plot(ref['v'], ref['t_ms'], '+', ms=12, mew=2,
                color='white' if ref['kind'] == 'primary' else 'magenta')
    if len(r['t_ms']):
        ax.plot(r['v'], r['t_ms'], '-', color='white', lw=0.8, alpha=0.7)
    for t, v, kind in zip(r['t_ms'], r['v'], r['match']['kind']):
        ax.plot(v, t, ls='none', mec='black', **STYLE[kind])
    m = r['match']
    ax.set_xlim(VEL_MIN, V_SHOW)
    ax.set_xlabel('Velocity (m/s)')
    return m


fig, axes = plt.subplots(len(MODES), len(CDP_LIST), figsize=(3.3 * len(CDP_LIST), 6.5 * len(MODES)),
                         sharey=True, squeeze=False)
for row, mode in enumerate(MODES):
    for col, x in enumerate(CDP_LIST):
        m = draw_case(axes[row, col], f'fd_layered_cdp{x:04d}_{mode}')
        dev = ' (dev)' if x == DEV_CDP else ''
        axes[row, col].set_title(f"CDP {x} m{dev}\nF1 {m['f1']:.2f}  K {m['n_picks']}  "
                                 f"mult. {m['n_on_multiple']}", fontsize=10)
    axes[row, 0].set_ylabel(f'{MODES[mode]}\nTime (ms)')
plt.suptitle(f'{ALG} — final picks on every CDP (G = {MIN_VEL_GRADIENT:g} m/s per s)', y=1.0)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'multicdp_{TAG}_panels.png'), dpi=110, bbox_inches='tight')
plt.show()

## 7. Results per CDP

F1 and recall per CDP (bars: with / without free surface), and the
summary table. The held-out CDPs are the ones to look at: a formulation
tuned on 2500 m that only works there would show it here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)
w = 0.38
xs = np.arange(len(CDP_LIST))
for k, metric in enumerate(['f1', 'recall']):
    for j, mode in enumerate(MODES):
        vals = [results[f'fd_layered_cdp{x:04d}_{mode}']['match'][metric] for x in CDP_LIST]
        axes[k].bar(xs + (j - 0.5) * w, vals, w, label=MODES[mode])
    axes[k].set_xticks(xs)
    axes[k].set_xticklabels([f'{x}' + ('\n(dev)' if x == DEV_CDP else '') for x in CDP_LIST])
    axes[k].set_ylim(0, 1.2); axes[k].set_ylabel(metric.upper() if metric == 'f1' else 'Recall')
    axes[k].set_xlabel('CDP (m)'); axes[k].grid(True, axis='y', alpha=0.3)
axes[0].legend(fontsize=8, loc='upper left', ncol=2)
plt.suptitle(f'{ALG} — scores per CDP')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'multicdp_{TAG}_scores.png'), dpi=130)
plt.show()

print(f"{'case':26s} {'f':>7s} {'K':>3s} {'prim':>6s} {'mult':>5s} {'P':>5s} {'R':>5s} {'F1':>5s}")
for name in CASES:
    r, m = results[name], results[name]['match']
    print(f"{name:26s} {r['score']:7.3f} {m['n_picks']:3d} {m['n_found']:3d}/{m['n_primaries']:<2d} "
          f"{m['n_on_multiple']:5d} {m['precision']:5.2f} {m['recall']:5.2f} {m['f1']:5.2f}")
for mode in MODES:
    for label, sel in [('all CDPs', CDP_LIST), ('held-out CDPs', [x for x in CDP_LIST if x != DEV_CDP])]:
        f1 = [results[f'fd_layered_cdp{x:04d}_{mode}']['match']['f1'] for x in sel]
        if f1:
            print(f"{mode:5s} {label:14s}: mean F1 {np.mean(f1):.2f}  "
                  f"(min {np.min(f1):.2f}, max {np.max(f1):.2f}, {len(f1)} CDPs)")

## 8. NMO check: gathers before and after correction

The qualitative check an interpreter makes: the picks define a velocity
function (linear between picks, constant above the first and below the
last) and the gather is NMO-corrected with it. One row per CDP, one
figure per mode:

1. **gather** with the moveout curve of every pick (cyan = on a primary,
   magenta = on a multiple, red = on neither);
2. **NMO with the picked velocities**: a primary picked at the right
   velocity becomes **flat**; picked too slow it curves **up**
   (over-corrected), too fast it curves **down**. An event that was not
   picked takes the velocity interpolated from its neighbors — a missed
   primary stays slightly curved, a multiple below a primary curves down
   (it is slower than the primary trend);
3. **NMO with the reference** (semblance peaks of the primaries only):
   what the corrected gather should look like — the primaries flat and
   the multiples still curved;
4. **stacks** of both (same scale): a flat event stacks into a strong,
   sharp wavelet; a curved one partly cancels.

Samples stretched by more than `NMO_STRETCH_MUTE` are muted, as in the
objective. This check is not part of the search: the semblance already
measures the same flatness pick by pick, but here the whole velocity
function is checked at once, including the interpolation between picks.

In [ ]:
sel = CDP_LIST if NMO_CDPS is None else [x for x in NMO_CDPS if x in CDP_LIST]
for mode in MODES:
    fig, axes = plt.subplots(len(sel), 4, figsize=(15, 4.8 * len(sel)), squeeze=False,
                             gridspec_kw={'width_ratios': [1, 1, 1, 0.45]})
    for row, x in enumerate(sel):
        name = f'fd_layered_cdp{x:04d}_{mode}'
        d, r = data[name], results[name]
        plot_nmo_check(axes[row], d['gather'], d['offsets'], d['dt'], r['t_ms'], r['v'],
                       reference=d['reference'], kinds=r['match']['kind'],
                       stretch_mute=NMO_STRETCH_MUTE)
        dev = ' (dev)' if x == DEV_CDP else ''
        axes[row, 0].set_ylabel(f"CDP {x} m{dev}  —  F1 {r['match']['f1']:.2f}\nTime (ms)")
    plt.suptitle(f'{ALG} — NMO check, {MODES[mode]}', y=1.0)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'multicdp_{TAG}_nmo_{mode}.png'), dpi=100, bbox_inches='tight')
    plt.show()

## 9. Picks along the line

Time of every pick against the CDP position, as an interpreter would see
a horizon across the line. Lines: reference events (solid white-edged
black = primaries, dashed magenta = surface multiples), joined from CDP
to CDP; markers: the algorithm's picks (same symbols as in section 6).
Real horizons are continuous: a pick that jumps between neighbors, or a
primary missing on one CDP but found on both sides, is what a lateral
continuity term (a later step) would correct.

In [ ]:
fig, axes = plt.subplots(1, len(MODES), figsize=(8 * len(MODES), 7), sharey=True, squeeze=False)
for ax, mode in zip(axes[0], MODES):
    names = [f'fd_layered_cdp{x:04d}_{mode}' for x in CDP_LIST]
    events = {}
    for x, name in zip(CDP_LIST, names):
        for ref in data[name]['reference']:
            events.setdefault((ref['kind'], ref['name']), []).append((x, ref['t_ms']))
    for (kind, ev), pts in events.items():
        pts = np.array(sorted(pts))
        if kind == 'primary':
            ax.plot(pts[:, 0], pts[:, 1], '-', color='black', lw=1.2)
            ax.text(pts[-1, 0] + 40, pts[-1, 1], ev, fontsize=8, va='center')
        else:
            ax.plot(pts[:, 0], pts[:, 1], '--', color='magenta', lw=0.9, alpha=0.7)
    for x, name in zip(CDP_LIST, names):
        r = results[name]
        for t, kind in zip(r['t_ms'], r['match']['kind']):
            st = STYLE[kind]
            ax.plot(x, t, ls='none', marker=st['marker'], ms=st['ms'] + 1,
                    mfc=st['color'], mec='black')
    for xf in faults:
        ax.axvline(xf, color='red', lw=1.0, alpha=0.6)
    ax.axvspan(DEV_CDP - 60, DEV_CDP + 60, color='gray', alpha=0.15)
    ax.set_xlim(min(CDP_LIST) - 200, max(CDP_LIST) + 400)
    ax.set_ylim(data[names[0]]['t_grid'][-1], 0)
    ax.set_xlabel('CDP position (m)'); ax.set_title(f'{ALG} — {MODES[mode]}')
    ax.grid(True, alpha=0.3)
axes[0, 0].set_ylabel('Time (ms)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f'multicdp_{TAG}_line.png'), dpi=130)
plt.show()

## 10. Summary

**What to look for:**
- whether the picks found on the development CDP (2500 m) carry over to
  the held-out CDPs, or whether the formulation (semblance options, tau,
  separation, minimum gradient) only fits that one gather;
- the CDPs next to the fault (3200 and 3400 m): diffractions from the
  fault edge add coherent energy that is neither a primary nor a surface
  multiple, and the event times jump across the fault;
- with free surface, how many CDPs end with a pick on a multiple
  (magenta triangle), with and without the minimum gradient
  (`MIN_VEL_GRADIENT` = 0 or 200);
- the Memetic Algorithm combines solutions by time cuts: check whether it
  keeps the intermediate primaries that Tabu Search sometimes drops, and
  whether the population (several solutions at once) avoids the CDP-to-CDP
  scatter of a single trajectory;
- in the NMO check, events that are not flat after the correction with
  the picks but are flat with the reference: a wrong or missing pick;
  and multiples flattened by the picks (a pick on a multiple);
- in the along-the-line view, primaries missing on isolated CDPs: the
  information to recover them is in the neighbors, which a lateral
  continuity term would use.

Same seed and budget on every case, one run each: differences between
CDPs can be luck of a single run. The batch configs give 10 runs per case
and the statistical tests (including pooled tests per group: with / without
free surface, held-out CDPs).